# U-Net do zero no CamVid — estudo de ablação

Roteiro (execute na ordem):
1. Ambiente e GPU · 2. Dados · 3. Exploração (figura de classes) · 4. Smoke test (tempo por época)
5. **Primeiros resultados**: baseline, seed 0 · 6. Plano completo (variantes × seeds) · 7. Tabelas e figuras

Ambiente: *Runtime → Change runtime type → GPU (L4 recomendada)*.

## 1. Ambiente

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/unet-camvid'   # resultados e checkpoints persistem no Drive
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
!git clone https://github.com/ereami/unet-camvid.git /content/unet-camvid
%cd /content/unet-camvid

## 2. Dados (CamVid, divisão do SegNet: 367 / 101 / 233)

In [ ]:
from src.data import download_camvid, CamVid, CLASSES
DATA = download_camvid('/content/data')          # disco local do Colab, não o Drive
train_ds, val_ds, test_ds = CamVid(DATA, 'train'), CamVid(DATA, 'val'), CamVid(DATA, 'test')
DATASETS = (train_ds, val_ds, test_ds)           # carregado uma vez, reutilizado em todos os treinos
print(train_ds.x.shape, val_ds.x.shape, test_ds.x.shape)

## 3. Exploração: desbalanceamento de classes (figura para o relatório)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
cnt = np.bincount(train_ds.y.ravel(), minlength=12)[:11]
freq = 100 * cnt / cnt.sum()
order = np.argsort(freq)[::-1]
plt.figure(figsize=(6, 2.6))
plt.bar([CLASSES[i] for i in order], freq[order], color='#4c72b0')
plt.ylabel('% dos pixels (treino)'); plt.xticks(rotation=45, ha='right'); plt.yscale('log')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_classes.png', dpi=200)
dict(zip(CLASSES, freq.round(2)))

## 4. Smoke test — confere o pipeline e mede o tempo por época
Extrapole: tempo/época × 120 = tempo de um treino.

In [ ]:
from src.train import Config, run
smoke = run(Config(name='smoke', epochs=2, eval_every=1), DATA, '/content/smoke', DATASETS)

## 5. Primeiros resultados: baseline (seed 0)
Este é o resultado que entra na seção *Primeiros resultados* do relatório.

In [ ]:
base0 = run(Config(name='baseline', seed=0), DATA, OUT_DIR, DATASETS)
{k: base0[k] for k in ['params_M', 'best_val_mIoU', 'test_mIoU', 'test_pixel_acc', 'train_min']}

In [ ]:
from src.report import plot_curves, plot_examples
import torch
from src.model import UNet
plot_curves(OUT_DIR, variants=['baseline'], path=f'{OUT_DIR}/fig_curvas_baseline.png')

dev = torch.device('cuda')
m = UNet(base=32).to(dev)
m.load_state_dict(torch.load(f'{OUT_DIR}/runs/baseline_s0/best.pt'))
plot_examples(m, test_ds, [0, 60, 150], dev, path=f'{OUT_DIR}/fig_exemplos_baseline.png');

In [ ]:
import pandas as pd
pd.Series({c: base0[f'iou_{c}'] for c in CLASSES}).sort_values().round(3)   # IoU por classe no teste

## 6. Plano completo — variantes × seeds
Retomável: execuções concluídas são puladas; interrompidas continuam do último checkpoint.

In [ ]:
from src.experiments import configs
# Prioridade: Q1-Q3 primeiro; capacidade (base16/base64) se sobrar orçamento
PRIORITARIAS = ['baseline', 'no_skip', 'wce', 'ce_dice', 'no_aug']
for cfg in configs(PRIORITARIAS):
    run(cfg, DATA, OUT_DIR, DATASETS)

In [ ]:
# Opcional (Q4)
for cfg in configs(['base16', 'base64']):
    run(cfg, DATA, OUT_DIR, DATASETS)

## 7. Tabelas e figuras

In [ ]:
from src.report import summary_table, per_class_table
display(summary_table(OUT_DIR))
display(per_class_table(OUT_DIR))
plot_curves(OUT_DIR, path=f'{OUT_DIR}/fig_curvas_todas.png');